# INSTRUCTOR GUIDE: Week 6 Session 1 SQL fundamentals at Sunrise Stores
Not for students. Solutions are filled in below each Your turn. Keep `sunrise_stores.db` (and the five CSVs) in the same folder as this notebook.

## Run of show (2 hours)
| Time | Block | What happens |
|---|---|---|
| 0:00 to 0:10 | Check-in and mission | Wins and blockers from the statistics week. Confirm everyone can open today's folder in VS Code. Say plainly: we will use SQLite inside the notebook, so nobody needs to install a database server. |
| 0:10 to 0:30 | Concept: what SQL is, the shop, the five tables | Walk the schema on the board or the markdown table. Grain of each table. Why we do not start with SELECT *. |
| 0:30 to 0:55 | Live coding: connect, SELECT, WHERE, ORDER BY | Your turn 1 and 2. Pause until everyone has a result, not just a query typed. |
| 0:55 to 1:05 | Break | |
| 1:05 to 1:40 | Aggregations, GROUP BY, HAVING | The HAVING versus WHERE contrast is the main idea. Your turn 3 and 4. |
| 1:40 to 1:55 | Challenge | Three short questions. Share one answer each. |
| 1:55 to 2:00 | Recap and assignment | Practice queries this week; the full SQL Analytics Challenge is introduced properly in Session 2, but they can start reading the brief. |

**If time runs short:** skip Your turn 4 part 2; keep the HAVING demo and the challenge.

## Talking points
- SQL is how we ask a question of tables that live in a database. Pandas is still useful; today the tables are in SQLite and we query them with SQL, then display the result with pandas.
- Notion mentions PostgreSQL as a later option. For this course we stay on SQLite so the notebook runs locally with the Python standard library. The SQL we write (SELECT, WHERE, GROUP BY, HAVING, and later JOIN) is the same idea in both.
- A query answers one question. Read it from the inside: FROM the table, WHERE the rows we keep, then what we SELECT, then how we ORDER the result.
- `SELECT *` is fine for a first peek, not for a final answer.
- `WHERE` filters rows before any grouping. `HAVING` filters groups after `GROUP BY`. If they try `WHERE AVG(unit_price) > 2500`, the database will refuse it. That refusal is useful: talk through it rather than skipping it.
- Aliases (`AS n`) are just display names. They make results readable.

## Common mistakes to watch for
- `FileNotFoundError` or `unable to open database file`: the notebook is not running in the folder that holds `sunrise_stores.db`. In VS Code, open the Week-6 folder and run from there.
- `SELECT *` as the finished answer.
- Forgetting `GROUP BY` when using `COUNT` or `AVG` with a category column.
- Putting an aggregate in `WHERE` instead of `HAVING`.
- Using `and` / `or` like Python instead of `AND` / `OR` in SQL (SQL wants the uppercase keywords; unquoted `and` often still works, but teach `AND`).
- String quotes: SQL strings use single quotes (`'Lagos'`). Double quotes are for names, not values, in standard SQL.
- Running cells out of order: they need `conn` from the first code cells. Restart and Run All if it breaks.
- Reading cancelled orders as sales. Status matters; we will be strict about `Completed` from Session 2 onwards, and we practise the habit today.

## Expected numbers (Sunrise Stores)
- 80 customers, 31 products, 420 orders, 1,071 order items, 379 payments.
- Orders by status: Completed 358, Cancelled 41, Refunded 21.
- 31 customers in Lagos, 15 Ibadan, 15 Abuja. Cities with more than 10 customers: those three.
- 16 Lagos customers signed up from 1 January to 30 June 2026.
- Beverages, expensive first: Soft drink crate 4,500; Malt drink 6-pack 3,200; Instant coffee 2,800; Orange juice 1L 2,200; Bottled water pack 1,800; Tea bags 1,500.
- Categories with average price above 2,500: Staples 5,200.0 (11 products), Beverages 2,666.7 (6), Household 2,550.0 (6).


# Week 6 Session 1: SQL fundamentals at Sunrise Stores
**Forge Academy | Data Science Foundations**

*Covers Week 6 Session 1: SQL fundamentals. We use SQLite inside this notebook, so you do not need to install a database server.*

## Mission

Sunrise Stores is a small provisions shop. The owner, Ngozi, has been keeping five tables: customers, products, orders, order items and payments. She can open the files, but she cannot yet ask them a precise question, such as "how many of last season's orders were cancelled?" or "which product category has the highest average price?"

Today we learn enough SQL to ask those questions of one table at a time. Next session we will join the tables and start the SQL Analytics Challenge.

Keep `sunrise_stores.db` in the same folder as this notebook, open the folder in VS Code, and run the cells from top to bottom.


## 0. What SQL is, in plain words

- A **database** is a collection of tables. Each table has a name, columns, and rows, just like a CSV, but we talk to it with a language called **SQL** (Structured Query Language).
- A **query** is a question written in SQL. The most common pattern is: take rows `FROM` a table, keep only some of them with `WHERE`, and `SELECT` the columns (or the totals) you want to see, then `ORDER BY` the result.
- We will run SQL from this notebook with `sqlite3` (part of Python) and `pandas` (to display the result as a table). The ideas you learn will still hold if you later use another database tool.

The shop's five tables, and the grain of each one (what one row means):

| Table | One row is | Useful columns |
|---|---|---|
| `customers` | one person who has signed up | `customer_id`, `first_name`, `last_name`, `city`, `signup_date`, `segment` |
| `products` | one item on the shelf | `product_id`, `product_name`, `category`, `unit_price`, `unit_cost` |
| `orders` | one basket / checkout | `order_id`, `customer_id`, `order_date`, `status`, `channel` |
| `order_items` | one line on an order (a product and a quantity) | `item_id`, `order_id`, `product_id`, `quantity`, `unit_price` |
| `payments` | one payment against an order | `payment_id`, `order_id`, `payment_date`, `method`, `amount` |

`segment` is `New`, `Regular` or `Wholesale`. `channel` is `Store`, `WhatsApp` or `Website`. `status` is `Completed`, `Cancelled` or `Refunded`. We will treat **Completed** as a real sale when we talk about money.

The five tables are also saved as CSVs next to the database, if you ever want to peek with `pd.read_csv`. Today we query the database.


## 1. Connect to the database and look around

We import `sqlite3` and `pandas`, then open a connection to the file.


In [ ]:
import sqlite3


In [ ]:
import pandas as pd


In [ ]:
conn = sqlite3.connect("sunrise_stores.db")


If you get an error that the file could not be opened, the notebook is not running in the same folder as `sunrise_stores.db`. Move the file there, or in VS Code use **File, Open Folder** on the Week 6 folder, and run the cell again.

`sqlite_master` is a built-in list of everything in the database. This query asks for the table names.


In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;", conn)


In [ ]:
tables


A first peek at each table. `LIMIT 5` means 'only the first 5 rows', which is safer than asking for everything.


In [ ]:
pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM products LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM order_items LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM payments LIMIT 5;", conn)


How big is each table? `COUNT(*)` counts rows.


In [ ]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM payments;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** Expected counts: customers 80, products 31, orders 420, order_items 1,071, payments 379. `UNION ALL` stacks the five small results into one table; mention it as a convenience, not a new topic to practise today.


You should see 80 customers, 31 products, 420 orders, 1,071 order items and 379 payments. We will keep coming back to those counts. If a later query quietly changes the number of rows, something about the question (or a join, next session) needs a second look.


## 2. SELECT, ORDER BY and a first filter

You rarely need every column. Name the ones you want. `ORDER BY` sorts the result; add `DESC` for highest first.


In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
ORDER BY unit_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


`WHERE` keeps only the rows that match a condition. Text values go in single quotes. This lists the beverages, expensive first.


In [ ]:
query = """
SELECT product_name, unit_price
FROM products
WHERE category = 'Beverages'
ORDER BY unit_price DESC;
"""


In [ ]:
result = pd.read_sql(query, conn)


In [ ]:
result


**INSTRUCTOR NOTE.** Six rows: Soft drink crate 4500, Malt drink 6-pack 3200, Instant coffee 2800, Orange juice 1L 2200, Bottled water pack 1800, Tea bags 1500. Ask someone to read the most and least expensive aloud.


You can combine conditions with `AND` and `OR`, and compare numbers with `>`, `<`, `>=`, `<=`, `=`. `BETWEEN` is inclusive on both ends. `IN` checks a list of values.


In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
WHERE unit_price >= 4000
ORDER BY unit_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


In [ ]:
query = """
SELECT first_name, last_name, city, segment
FROM customers
WHERE city IN ('Lagos', 'Ibadan')
  AND segment = 'Wholesale'
ORDER BY city, last_name;
"""


In [ ]:
pd.read_sql(query, conn)


Dates in this database are stored as text in `YYYY-MM-DD` form, so we can compare them as text and the order will still be chronological.


In [ ]:
query = """
SELECT first_name, last_name, city, signup_date
FROM customers
WHERE signup_date >= '2026-01-01'
ORDER BY signup_date
LIMIT 8;
"""


In [ ]:
pd.read_sql(query, conn)


### Your turn 1

1. List every product in the `Staples` category. Show `product_name` and `unit_price`, cheapest first.
2. How many customers live in Lagos? (Hint: `COUNT(*)` and a `WHERE` on `city`.)


In [ ]:
# SOLUTION
query = """
SELECT product_name, unit_price
FROM products
WHERE category = 'Staples'
ORDER BY unit_price;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION
query = """
SELECT COUNT(*) AS n_lagos
FROM customers
WHERE city = 'Lagos';
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 1.** Staples, cheapest first, starts with Bread loaf 1,200 and ends with Rice 10kg 15,500 and Premium olive oil 1L 9,800. Lagos has **31** customers. If someone gets 0, they probably wrote `lagos` in lower case; these values are title case.


### Your turn 2

Show `customer_id`, `first_name`, `last_name`, `signup_date` and `segment` for customers who live in Lagos and who signed up on or after 1 January 2026 and before 1 July 2026. Order the rows by `signup_date`.


In [ ]:
# SOLUTION
query = """
SELECT customer_id, first_name, last_name, signup_date, segment
FROM customers
WHERE city = 'Lagos'
  AND signup_date >= '2026-01-01'
  AND signup_date < '2026-07-01'
ORDER BY signup_date;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 2.** **16 rows.** First is customer 51, Ruth Ibe, 2026-02-04, Regular. Last is customer 75, Olu Abdullahi, 2026-06-25, Wholesale. If they used `<= '2026-06-30'` that is also correct. If they used `BETWEEN '2026-01-01' AND '2026-07-01'` they would accidentally include 1 July, had anyone signed up that day.


## 3. Totals: COUNT, SUM, AVG, MIN, MAX

With no `GROUP BY`, an aggregate function summarises the **whole** (filtered) table into one row.


In [ ]:
query = """
SELECT
    COUNT(*) AS n_orders,
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order
FROM orders;
"""


In [ ]:
pd.read_sql(query, conn)


How many orders are there in each status? We will group in a moment. First, filter to one status at a time so the idea of a subset is clear.


In [ ]:
query = """
SELECT COUNT(*) AS n_completed
FROM orders
WHERE status = 'Completed';
"""


In [ ]:
pd.read_sql(query, conn)


On the products table, what is the average, cheapest and most expensive unit price?


In [ ]:
query = """
SELECT
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price,
    MIN(unit_price) AS min_price,
    MAX(unit_price) AS max_price
FROM products;
"""


In [ ]:
pd.read_sql(query, conn)


`ROUND(..., 1)` keeps one decimal place so the average is readable. `DISTINCT` counts unique values only. This is how many cities appear in the customer list, and how many different segments.


In [ ]:
query = """
SELECT
    COUNT(*) AS n_customers,
    COUNT(DISTINCT city) AS n_cities,
    COUNT(DISTINCT segment) AS n_segments
FROM customers;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** 420 orders, 358 completed, dates from 2026-01-01 through 2026-10-31. 80 customers, 7 cities, 3 segments. Product prices: min 800 (Sponge pack), max 15,500 (Rice 10kg).


## 4. GROUP BY: one total per group

`GROUP BY city` says: split the customers into a pile per city, then count each pile. Every column in the `SELECT` list must either be in the `GROUP BY` or be an aggregate (`COUNT`, `SUM`, `AVG`, ...).


In [ ]:
query = """
SELECT city, COUNT(*) AS n_customers
FROM customers
GROUP BY city
ORDER BY n_customers DESC;
"""


In [ ]:
pd.read_sql(query, conn)


The same idea, for orders by status and for products by category.


In [ ]:
query = """
SELECT status, COUNT(*) AS n_orders
FROM orders
GROUP BY status
ORDER BY n_orders DESC;
"""


In [ ]:
pd.read_sql(query, conn)


In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price,
    MIN(unit_price) AS min_price,
    MAX(unit_price) AS max_price
FROM products
GROUP BY category
ORDER BY avg_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** Cities: Lagos 31, Ibadan 15, Abuja 15, Port Harcourt 8, Enugu 5, Kano 3, Abeokuta 3. Status: Completed 358, Cancelled 41, Refunded 21. Category average prices: Staples 5,200.0 (11), Beverages 2,666.7 (6), Household 2,550.0 (6), Personal Care 1,525.0 (4), Snacks 1,075.0 (4). If someone forgets `GROUP BY category` they will get an error. Show the error, then add the clause. That is more memorable than only showing the finished query.


### Your turn 3

1. Count the customers in each `segment`. Show the segment and the count, largest group first.
2. Count the orders in each `channel`. Show the channel and the count, largest first.


In [ ]:
# SOLUTION
query = """
SELECT segment, COUNT(*) AS n_customers
FROM customers
GROUP BY segment
ORDER BY n_customers DESC;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION
query = """
SELECT channel, COUNT(*) AS n_orders
FROM orders
GROUP BY channel
ORDER BY n_orders DESC;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 3.** Segments: Regular 37, New 28, Wholesale 15. Channels (all statuses): Store 232, WhatsApp 103, Website 85. If a student filters to Completed here, they will get 194 / 92 / 72 instead. That is not wrong thinking, but the question asked for all orders; praise the care and ask them to drop the filter.


## 5. WHERE versus HAVING

This is the idea that trips people up most often, so we will go slowly.

- `WHERE` filters **rows**, before the groups are built.
- `HAVING` filters **groups**, after `GROUP BY` has produced them.

Example: "which categories have an average unit price above 2,500?" The average does not exist until after we group, so the condition belongs in `HAVING`.


In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2500
ORDER BY avg_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


If you move that condition into `WHERE`, SQLite will raise an error, because `WHERE` cannot see `AVG(unit_price)` yet. That error is a useful one. The rule of thumb: if the condition is about a total or an average of a group, it is a `HAVING`. If it is about a single row (this product is a staple; this order is completed), it is a `WHERE`.

You will often use both in the same query. `WHERE` first narrows the rows, then `GROUP BY` builds the piles, then `HAVING` keeps some of the piles.


In [ ]:
query = """
SELECT city, COUNT(*) AS n_customers
FROM customers
WHERE segment = 'Regular'
GROUP BY city
HAVING COUNT(*) >= 3
ORDER BY n_customers DESC;
"""


In [ ]:
pd.read_sql(query, conn)


**INSTRUCTOR NOTE.** Categories above 2,500: Staples 5,200.0, Beverages 2,666.7, Household 2,550.0. Snacks and Personal Care drop out. Optionally type the wrong `WHERE AVG(unit_price) > 2500` version on a scratch cell, show the error, then delete it so a Run All still works. The Regular-and-at-least-3-cities query is just a pattern demo; do not spend long on the exact numbers.


### Your turn 4

1. Which cities have **more than 10** customers? Show the city and the count, largest first.
2. Looking only at **Completed** orders, count the orders in each channel and keep channels that have **more than 80** completed orders. (You will need both `WHERE` and `HAVING`.)


In [ ]:
# SOLUTION
query = """
SELECT city, COUNT(*) AS n_customers
FROM customers
GROUP BY city
HAVING COUNT(*) > 10
ORDER BY n_customers DESC;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION
query = """
SELECT channel, COUNT(*) AS n_completed
FROM orders
WHERE status = 'Completed'
GROUP BY channel
HAVING COUNT(*) > 80
ORDER BY n_completed DESC;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Your turn 4.** Cities: Lagos 31, Ibadan 15, Abuja 15. Completed channels with more than 80: Store 194 and WhatsApp 92. Website has 72 completed orders, so it should drop out. If Website still appears, they used `>= 80` or forgot `WHERE status = 'Completed'` (Website has 85 orders of any status).


## 6. Challenge (10 to 15 minutes)

Write three queries, one for each question. Use only the single tables we have already used. No joins yet.

1. List every Staples product that costs **3,000 or more**. Show `product_name` and `unit_price`, most expensive first.
2. How many orders were placed in each `status`? You already saw this pattern; write it yourself from scratch.
3. Which product categories have **more than 4** products? Show the category, the number of products, and the average `unit_price` rounded to one decimal, largest average first.


In [ ]:
# SOLUTION — question 1
query = """
SELECT product_name, unit_price
FROM products
WHERE category = 'Staples'
  AND unit_price >= 3000
ORDER BY unit_price DESC;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION — question 2
query = """
SELECT status, COUNT(*) AS n_orders
FROM orders
GROUP BY status
ORDER BY n_orders DESC;
"""
pd.read_sql(query, conn)


In [ ]:
# SOLUTION — question 3
query = """
SELECT
    category,
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price
FROM products
GROUP BY category
HAVING COUNT(*) > 4
ORDER BY avg_price DESC;
"""
pd.read_sql(query, conn)


**INSTRUCTOR NOTE. Challenge.**

1. Rice 10kg 15,500; Premium olive oil 1L 9,800; Rice 5kg 8,200; Indomie carton 6,800; Vegetable oil 2L 4,200; Beans 2kg 3,600. Six rows. If olive oil is missing, they used `>` instead of `>=`? No, 9800 is well above 3000. If rice 5kg is missing they used a higher cutoff.
2. Completed 358, Cancelled 41, Refunded 21.
3. Staples 11 products, avg 5,200.0; Beverages 6, avg 2,666.7; Household 6, avg 2,550.0. Snacks and Personal Care have 4 products each, so they should drop out (`>` not `>=`).


## Recap

- A SQL query is a precise question: `SELECT` what you want, `FROM` a table, `WHERE` the rows you keep, `GROUP BY` when you want one total per group, `HAVING` when you filter those groups, `ORDER BY` to sort, `LIMIT` to peek.
- `COUNT`, `SUM`, `AVG`, `MIN` and `MAX` summarise. With no `GROUP BY`, they summarise the whole filtered table. With `GROUP BY`, they summarise each pile.
- `WHERE` filters rows; `HAVING` filters groups. If the condition uses an aggregate, it belongs in `HAVING`.
- `SELECT *` is a peek, not a finished answer. Name the columns you mean to show.
- We used SQLite through `sqlite3` and displayed results with `pd.read_sql`. Keep `conn` from the first cells; if something breaks, restart the kernel and Run All.

Next session we join these tables, check that we have not accidentally doubled any rows, and start the SQL Analytics Challenge. We will also look at the capstone brief, so you can start thinking about a problem and a dataset.


## Assignment (practice, before Session 2)

**Due:** before our next class. **Submit:** a short `.sql` file or a notebook, through the usual submission form.

Write numbered queries against `sunrise_stores.db`. Add a one-line comment above each query saying what it answers. Use `SELECT`, `WHERE`, `GROUP BY` and `HAVING` as needed. Do not join tables yet.

1. List every product in `Personal Care`, cheapest first.
2. Count the customers in each city, largest first.
3. How many orders were `Cancelled`?
4. Which segments have **more than 20** customers? Show the segment and the count.
5. Among products that cost **less than 2,000**, what is the average `unit_price` in each category? (Filter rows with `WHERE`, then group.)

**Rubric (20 points, 0 to 4 each)**

| Area | What earns full marks |
|---|---|
| Correctness | Queries run and the numbers are right |
| Problem solving | `WHERE` and `HAVING` are used in the right places |
| Code quality | Numbered queries, short comments, no leftover `SELECT *` |
| Communication | Each result can be read as a plain-language answer |
| Independence | You can explain every line on our call |

AI tools are allowed, but you must be able to explain every line of your work on our call.

A longer **SQL Analytics Challenge** (joins plus a write-up, 10% of the course) will be set properly in Session 2. If you want a preview, open `Week6_SQL_Analytics_Challenge.txt` in this folder, but do not worry about finishing it yet.


**INSTRUCTOR NOTE. Practice assignment answers.**

1. Toothbrush 2-pack 900; Hand sanitiser 1,200; Toothpaste 1,600; Body cream 2,400. Four rows, cheapest first.
2. Lagos 31, Ibadan 15, Abuja 15, Port Harcourt 8, Enugu 5, Kano 3, Abeokuta 3.
3. 41 cancelled orders.
4. Regular 37 and New 28. Wholesale has 15, so it drops out.
5. Products under 2,000, average by category (rounded later is fine):

Run this if you want to confirm in class:

```
SELECT category, COUNT(*) AS n, ROUND(AVG(unit_price), 1) AS avg_price
FROM products
WHERE unit_price < 2000
GROUP BY category
ORDER BY avg_price;
```

Personal Care 1,233.3 (3 products: 900, 1,200, 1,600 — body cream 2,400 is excluded); Household 1,366.7 (800, 1,400, 1,900 — detergent 1,900, bar soap 1,400, sponge 800; toilet paper 2,600 and disinfectant 2,100 and ceramic jar 6,500 drop out); Beverages 1,650.0 (tea 1,500, water 1,800); Snacks 1,075.0 (all four snacks are under 2,000); Staples 1,533.3 (bread 1,200, sugar 1,600, garri 1,800).
